# Follow-Up Adapter — Fine-tuning

Trains the model to ask **one focused follow-up question** per turn given a
patient complaint and conversation history.

**Datasets used:**
- `UCSD26/medical_dialog` (processed.en) — multi-turn doctor-patient conversations
- `lavita/ChatDoctor-HealthCareMagic-100k` — doctor responses with embedded questions

**Output:** `follow_up_adapter/` folder saved to Google Drive


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'unsloth[colab-new]', 'peft', 'accelerate',
    'bitsandbytes', 'trl', 'datasets', '-q'
], check=True)
print(' Installed.')

In [ ]:
# ── Mount Drive & set paths ───────────────────────────────────────────────────
from google.colab import drive
import os, json, re, random, glob, gc
import torch

drive.mount('/content/drive', force_remount=False)

# WHERE THE ADAPTER WILL BE SAVED
SAVE_PATH  = '/content/drive/MyDrive/unsloth_training/follow_up_adapter'
CKPT_DIR   = '/content/drive/MyDrive/unsloth_training/follow_up_checkpoints'
CACHE_PATH = '/content/drive/MyDrive/unsloth_training/followup_dataset.json'

os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(CKPT_DIR,  exist_ok=True)

# ── Model config ──────────────────────────────────────────────────────────────
BASE_MODEL     = 'unsloth/llama-2-7b-bnb-4bit'
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT   = True

# ── LoRA config (must match diagnostic adapter exactly) ───────────────────────
LORA_R         = 16
LORA_ALPHA     = 16
LORA_DROPOUT   = 0
TARGET_MODULES = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']

# ── Training config ───────────────────────────────────────────────────────────
BATCH_SIZE    = 2
GRAD_ACCUM    = 8
EPOCHS        = 1
LR            = 2e-4
SEED          = 3407
SAVE_STEPS    = 100

# ── Dataset caps ──────────────────────────────────────────────────────────────
MAX_MEDDIALOG  = 30_000
MAX_CHATDOCTOR = 20_000

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f' Config ready. GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'   Adapter will be saved to: {SAVE_PATH}')

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    'You are a medical intake doctor. Based on the patient complaint '
    'and conversation history below, ask ONE focused follow-up question. '
    'Do not diagnose. Ask only one question.'
)

def extract_first_question(text: str) -> str:
    m = re.search(r'[^.!\n]*\?', text)
    return m.group(0).strip() if m else text.split('\n')[0].strip()

def is_valid_question(text: str) -> bool:
    t = text.strip()
    return 20 <= len(t) <= 300 and '?' in t

def make_example(complaint: str, history: list, question: str) -> dict | None:
    q = extract_first_question(question)
    if not is_valid_question(q):
        return None
    hist_str = '\n'.join(f'{r}: {t}' for r, t in history) if history else '(none)'
    instruction = (
        f'{SYSTEM_PROMPT}\n\n'
        f'Patient complaint: {complaint.strip()}\n'
        f'Conversation so far:\n{hist_str}'
    )
    return {
        'text': f'### Instruction:\n{instruction}\n\n### Response:\n{q}'
    }

print(' Helpers defined.')

In [ ]:
# ── Build dataset (or load from cache) ───────────────────────────────────────
from datasets import load_dataset, Dataset

if os.path.exists(CACHE_PATH):
    print(f' Loading cached dataset from Drive...')
    with open(CACHE_PATH) as f:
        all_examples = json.load(f)
    print(f'  {len(all_examples):,} examples loaded.')

else:
    all_examples = []

    # ── MedDialog ────────────────────────────────────────────────────────────
    print(' Loading MedDialog...')
    try:
        ds = load_dataset('UCSD26/medical_dialog', 'processed.en', trust_remote_code=True)
        md_ex = []
        for row in ds['train']:
            if len(md_ex) >= MAX_MEDDIALOG:
                break
            utts      = row.get('utterances', [])
            complaint = row.get('description', '').strip()
            if not utts or not complaint:
                continue
            history = []
            for i, utt in enumerate(utts):
                spk = 'Patient' if i % 2 == 0 else 'Doctor'
                if spk == 'Doctor':
                    ex = make_example(complaint, history.copy(), utt)
                    if ex:
                        md_ex.append(ex)
                history.append((spk, utt[:200]))
        all_examples.extend(md_ex)
        print(f'    MedDialog    → {len(md_ex):,} examples')
    except Exception as e:
        print(f'     MedDialog failed: {e}')

    # ── ChatDoctor ───────────────────────────────────────────────────────────
    print(' Loading ChatDoctor...')
    try:
        ds = load_dataset('lavita/ChatDoctor-HealthCareMagic-100k', trust_remote_code=True)
        cd_ex = []
        for row in ds['train']:
            if len(cd_ex) >= MAX_CHATDOCTOR:
                break
            complaint = row.get('input',  '').strip()
            response  = row.get('output', '').strip()
            if complaint and '?' in response:
                ex = make_example(complaint, [], response)
                if ex:
                    cd_ex.append(ex)
        all_examples.extend(cd_ex)
        print(f'    ChatDoctor   → {len(cd_ex):,} examples')
    except Exception as e:
        print(f'     ChatDoctor failed: {e}')

    random.seed(SEED)
    random.shuffle(all_examples)

    with open(CACHE_PATH, 'w') as f:
        json.dump(all_examples, f, ensure_ascii=False)
    print(f'\n Cached → {CACHE_PATH}')

print(f'\n Total training examples: {len(all_examples):,}')

# Show one sample
print('\n── Sample training example ──')
print(all_examples[0]['text'][:600])

hf_dataset = Dataset.from_list(all_examples)
print(f'\n Dataset ready: {len(hf_dataset):,} rows')

In [ ]:
# ── Load model + attach LoRA ──────────────────────────────────────────────────
from unsloth import FastLanguageModel

print(' Loading base model...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = BASE_MODEL,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = None,
    load_in_4bit  = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                         = LORA_R,
    target_modules            = TARGET_MODULES,
    lora_alpha                = LORA_ALPHA,
    lora_dropout              = LORA_DROPOUT,
    bias                      = 'none',
    use_gradient_checkpointing= 'unsloth',
    random_state              = SEED,
)
tokenizer.padding_side = 'right'

tp = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f' Model ready. Trainable params: {tp/1e6:.2f}M')

In [ ]:
# ── Resume from checkpoint (optional — skip if training fresh) ────────────────
ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, 'checkpoint-*')),
               key=lambda p: int(p.split('-')[-1]))
RESUME_FROM = ckpts[-1] if ckpts else None

if RESUME_FROM:
    print(f'  Resuming from: {RESUME_FROM}')
else:
    print(' Training from scratch.')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = hf_dataset,
    args          = SFTConfig(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = 5,
        num_train_epochs            = EPOCHS,
        learning_rate               = LR,
        lr_scheduler_type           = 'linear',
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        logging_steps               = 20,
        output_dir                  = CKPT_DIR,
        save_strategy               = 'steps',
        save_steps                  = SAVE_STEPS,
        save_total_limit            = 3,
        report_to                   = 'none',
        dataset_text_field          = 'text',
        max_seq_length              = MAX_SEQ_LENGTH,
        packing                     = False,
        seed                        = SEED,
    ),
)

print(f' Training follow_up_adapter on {len(hf_dataset):,} examples...')
result = trainer.train(resume_from_checkpoint=RESUME_FROM)
print(f'\n Done! Loss: {result.training_loss:.4f} | Steps: {result.global_step}')

In [ ]:
# ── Save adapter to Drive ─────────────────────────────────────────────────────
print(f' Saving follow_up_adapter to: {SAVE_PATH}')
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print('\n Saved files:')
for f in sorted(os.listdir(SAVE_PATH)):
    sz = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1e6
    print(f'   {f:45s}  {sz:.1f} MB')
print(f'\n Full path: {SAVE_PATH}')

In [ ]:
# ── Quick test ────────────────────────────────────────────────────────────────
FastLanguageModel.for_inference(model)

COMPLAINT = 'I have been having a severe headache for the past two days.'

prompt = (
    f'### Instruction:\n{SYSTEM_PROMPT}\n\n'
    f'Patient complaint: {COMPLAINT}\n'
    f'Conversation so far:\n(none)\n\n'
    f'### Response:\n'
)

inputs = tokenizer([prompt], return_tensors='pt').to(DEVICE)
with torch.no_grad():
    out = model.generate(
        **inputs, max_new_tokens=80, temperature=0.7,
        do_sample=True, pad_token_id=tokenizer.eos_token_id
    )
new = out[0][inputs['input_ids'].shape[1]:]
print('Patient:', COMPLAINT)
print('Doctor :', tokenizer.decode(new, skip_special_tokens=True).strip())

In [ ]:
# ── Free VRAM ─────────────────────────────────────────────────────────────────
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print(' VRAM freed. Run finetune_diagnostic.ipynb next.')